<a href="https://colab.research.google.com/github/nmatthews5/GithubAction/blob/master/Version_2_Final_March_Madness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [131]:
import pandas as pd
import pickle

# Load the main dataset
df = pd.read_csv('/content/KenPom Barttorvik.csv')

print('Main statistics dataset (df) loaded:')
display(df.head())

Main statistics dataset (df) loaded:


,YEAR,CONF,CONF ID,QUAD NO,QUAD ID,TEAM NO,TEAM ID,TEAM,SEED,ROUND,...,BADJT RANK,AVG HGT RANK,EFF HGT RANK,EXP RANK,TALENT RANK,FT% RANK,OP FT% RANK,PPPO RANK,PPPD RANK,ELITE SOS RANK
0,2026,MAC,17,69,1,1215,2,Akron,12,0,...,34,362,363,38,164,73,174,6,86,263
1,2026,SEC,28,69,1,1214,3,Alabama,4,0,...,4,89,76,294,8,42,47,5,249,6
2,2026,B12,7,70,2,1213,8,Arizona,1,0,...,43,8,26,324,9,158,130,15,5,10
3,2026,SEC,28,70,2,1212,10,Arkansas,4,0,...,23,30,49,309,6,99,162,4,201,13
4,2026,B12,7,70,2,1211,25,BYU,6,0,...,58,70,98,307,48,100,117,29,124,15


In [132]:
# Define first round matchups (Seed vs Seed)
first_round_matchups = {
    1: 16,
    8: 9,
    5: 12,
    4: 13,
    6: 11,
    3: 14,
    7: 10,
    2: 15
}
print('First round matchups defined.')

First round matchups defined.


### Training a New Logistic Regression Model
Since the pre-trained model could not be loaded, we will train a new model using historical data from the `df` DataFrame. The goal is to predict whether a team wins its first tournament game based on its statistics.

In [133]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Map seeds to historical win probabilities based on the upset data provided
seed_win_map = {
    1: 0.987, 16: 0.013, 2: 0.949, 15: 0.051,
    3: 0.859, 14: 0.141, 4: 0.795, 13: 0.205,
    5: 0.654, 12: 0.346, 6: 0.609, 11: 0.391,
    7: 0.603, 10: 0.397, 8: 0.500, 9: 0.500
}

# 1. Prepare Training Data
# Use 2026 as the threshold since that is the bracket year
BRACKET_YEAR = 2026
training_data = df[df['YEAR'] < BRACKET_YEAR].copy()

# Add the historical upset probability as a feature
training_data['HIST_PROB'] = training_data['SEED'].map(seed_win_map)

# Update Target: ROUND < 64 means they won at least one game
training_data['WON_FIRST_GAME'] = (training_data['ROUND'] < 64).astype(int)

# Updated features including the historical probability
model_features = ['SEED', 'HIST_PROB', 'KADJ O RANK', 'KADJ D RANK', '3PT%', 'AST%', 'TOV%', 'TALENT', 'ELITE SOS RANK', 'BARTHAG', 'EXP']
X = training_data[model_features].copy()
y = training_data['WON_FIRST_GAME']

X.dropna(inplace=True)
y = y.loc[X.index]

print(f'Training data prepared. Shape: {X.shape}')

Training data prepared. Shape: (1147, 11)


In [134]:
# 2. Split Data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')

X_train shape: (917, 11)
y_train shape: (917,)


In [135]:
# 3. Train a new Logistic Regression Model
log_reg_updated = LogisticRegression(random_state=42, solver='liblinear')
log_reg_updated.fit(X_train, y_train)

print('New Logistic Regression Model trained successfully.')

# Evaluate the new model
from sklearn.metrics import accuracy_score

y_pred = log_reg_updated.predict(X_test)
print(f'Model Accuracy: {accuracy_score(y_test, y_pred):.4f}')

New Logistic Regression Model trained successfully.
Model Accuracy: 0.7304


In [146]:
#Import Bracket
import pandas as pd
import io

bracket_data_str = """
Region,Seed,Team,Record,PlayIn,PlayInOpponent
EAST,1,Duke,32-2,No,
EAST,16,Siena,23-11,No,
EAST,8,Ohio State,21-12,No,
EAST,9,TCU,22-11,No,
EAST,5,St. John's,28-6,No,
EAST,12,Northern Iowa,23-12,No,
EAST,4,Kansas,23-10,No,
EAST,13,Cal Baptist,25-8,No,
EAST,6,Louisville,23-10,No,
EAST,11,South Florida,25-8,No,
EAST,3,Michigan State,25-7,No,
EAST,14,North Dakota State,27-7,No,
EAST,7,UCLA,23-11,No,
EAST,10,UCF,21-11,No,
EAST,2,UConn,29-5,No,
EAST,15,Furman,22-12,No,

SOUTH,1,Florida,26-7,No,
SOUTH,16,Lehigh,18-16,Yes,PVAMU
SOUTH,16,PVAMU,?,Yes,Lehigh
SOUTH,8,Clemson,24-10,No,
SOUTH,9,Iowa,21-12,No,
SOUTH,5,Vanderbilt,26-8,No,
SOUTH,12,McNeese,28-5,No,
SOUTH,4,Nebraska,26-6,No,
SOUTH,13,Troy,22-11,No,
SOUTH,6,North Carolina,24-8,No,
SOUTH,11,VCU,27-7,No,
SOUTH,3,Illinois,24-8,No,
SOUTH,14,Penn,18-11,No,
SOUTH,7,Saint Mary's,27-5,No,
SOUTH,10,Texas A&M,21-11,No,
SOUTH,2,Houston,28-6,No,
SOUTH,15,Idaho,21-14,No,

WEST,1,Arizona,32-2,No,
WEST,16,Long Island,24-10,No,
WEST,8,Villanova,24-8,No,
WEST,9,Utah State,28-6,No,
WEST,5,Wisconsin,24-10,No,
WEST,12,High Point,30-4,No,
WEST,4,Arkansas,26-8,No,
WEST,13,Hawaii,24-8,No,
WEST,6,BYU,23-11,No,
WEST,11,Texas,18-14,No,
WEST,3,Gonzaga,30-3,No,
WEST,14,Kennesaw State,21-13,No,
WEST,7,Miami (FL),25-8,No,
WEST,10,Missouri,20-12,No,
WEST,2,Purdue,27-8,No,
WEST,15,Queens (NC),21-13,No,

MIDWEST,1,Michigan,31-3,No,
MIDWEST,16,Howard,23-10,Yes,UMBC
MIDWEST,16,UMBC,24-8,Yes,Howard
MIDWEST,8,Georgia,22-10,No,
MIDWEST,9,Saint Louis,28-5,No,
MIDWEST,5,Texas Tech,22-10,No,
MIDWEST,12,Akron,29-5,No,
MIDWEST,4,Alabama,23-9,No,
MIDWEST,13,Hofstra,24-10,No,
MIDWEST,6,Tennessee,22-11,No,
MIDWEST,11,SMU,20-13,Yes,Miami (OH)
MIDWEST,11,Miami (OH),31-1,Yes,SMU
MIDWEST,3,Virginia,29-5,No,
MIDWEST,14,Wright State,23-11,No,
MIDWEST,7,Kentucky,21-13,No,
MIDWEST,10,Santa Clara,26-8,No,
MIDWEST,2,Iowa State,27-7,No,
MIDWEST,15,Tennessee State,23-9,No,
"""

bracket_df = pd.read_csv(io.StringIO(bracket_data_str))
bracket_df['Team'] = bracket_df['Team'].str.strip()
print('Bracket data updated: Texas set as 11-seed in West.')

Bracket data updated: Texas set as 11-seed in West.


In [147]:
# 1. Identify matchups with Win Prob < 80%
interesting_matchups = final_winners_df[final_winners_df['Win_Prob'] < 0.8].copy()

# 2. Extract detailed stats for both teams in those matchups
detailed_analysis = []

for _, row in interesting_matchups.iterrows():
    seeds = [int(s) for s in row['Matchup'].split(' vs ')]
    s1, s2 = seeds[0], seeds[1]
    region = row['Region']

    # Get full stats for both teams from bracket_final_cleaned
    t1_stats = bracket_final_cleaned[(bracket_final_cleaned['Region'] == region) & (bracket_final_cleaned['Seed'] == s1)].iloc[0]
    t2_stats = bracket_final_cleaned[(bracket_final_cleaned['Region'] == region) & (bracket_final_cleaned['Seed'] == s2)].iloc[0]

    analysis = {
        'Matchup': f"{t1_stats['Team']} ({s1}) vs {t2_stats['Team']} ({s2})",
        'Win_Prob': f"{row['Win_Prob']:.2%}",
        'Winner': row['Winner'],
        'Favorite_3PT%': t1_stats['3PT%'],
        'Underdog_3PT%': t2_stats['3PT%'],
        'Favorite_TOV%': t1_stats['TOV%'],
        'Underdog_TOV%': t2_stats['TOV%'],
        'Fav_Off_Rank': t1_stats['KADJ O RANK'],
        'Und_Off_Rank': t2_stats['KADJ O RANK'],
        'Fav_Def_Rank': t1_stats['KADJ D RANK'],
        'Und_Def_Rank': t2_stats['KADJ D RANK']
    }
    detailed_analysis.append(analysis)

detailed_df = pd.DataFrame(detailed_analysis)

print('Deep Dive Analysis: High-Efficiency & Key Stats for Competitive Matchups (<80% Prob)')
display(detailed_df.sort_values(by='Win_Prob'))

# Highlight statistical advantages for underdogs
print('\nUnderdog Statistical Advantages in Competitive Matchups:')
for idx, row in detailed_df.iterrows():
    advantages = []
    if row['Underdog_3PT%'] > row['Favorite_3PT%']: advantages.append('Better 3PT%')
    if row['Underdog_TOV%'] < row['Favorite_TOV%']: advantages.append('Fewer Turnovers')
    if row['Und_Def_Rank'] < row['Fav_Def_Rank']: advantages.append('Higher Defensive Efficiency')

    if advantages:
        print(f"- {row['Matchup']}: Underdog has edge in {', '.join(advantages)}")

Deep Dive Analysis: High-Efficiency & Key Stats for Competitive Matchups (<80% Prob)


,Matchup,Win_Prob,Winner,Favorite_3PT%,Underdog_3PT%,Favorite_TOV%,Underdog_TOV%,Fav_Off_Rank,Und_Off_Rank,Fav_Def_Rank,Und_Def_Rank
13,Texas Tech (5) vs Akron (12),30.76%,Akron,39.3,38.5,15.9,15.5,12,54,33,113
10,BYU (6) vs Texas (11),36.28%,Texas,34.9,34.9,15.3,15.3,10,13,57,112
16,Kentucky (7) vs Santa Clara (10),37.08%,Santa Clara,34.1,34.9,15.0,15.6,39,23,27,82
6,North Carolina (6) vs VCU (11),37.38%,VCU,34.5,36.7,14.0,15.2,32,46,37,60
15,Tennessee (6) vs SMU (11),37.48%,SMU,33.4,37.4,17.1,15.9,37,26,15,91
3,UCLA (7) vs UCF (10),37.98%,UCF,38.2,36.2,13.4,15.7,22,40,53,101
11,Miami (FL) (7) vs Missouri (10),41.67%,Missouri,34.7,35.0,16.2,18.0,33,50,38,78
2,Louisville (6) vs South Florida (11),45.19%,South Florida,35.7,33.1,16.4,15.2,19,61,25,40
12,Georgia (8) vs Saint Louis (9),49.51%,Saint Louis,34.1,40.5,14.3,17.4,16,51,80,42
0,Ohio State (8) vs TCU (9),54.11%,TCU,36.0,33.1,15.4,15.5,17,81,54,22



Underdog Statistical Advantages in Competitive Matchups:
- Ohio State (8) vs TCU (9): Underdog has edge in Higher Defensive Efficiency
- Louisville (6) vs South Florida (11): Underdog has edge in Fewer Turnovers
- Clemson (8) vs Iowa (9): Underdog has edge in Better 3PT%
- North Carolina (6) vs VCU (11): Underdog has edge in Better 3PT%
- Saint Mary's (7) vs Texas A&M (10): Underdog has edge in Fewer Turnovers
- Miami (FL) (7) vs Missouri (10): Underdog has edge in Better 3PT%
- Georgia (8) vs Saint Louis (9): Underdog has edge in Better 3PT%, Higher Defensive Efficiency
- Texas Tech (5) vs Akron (12): Underdog has edge in Fewer Turnovers
- Alabama (4) vs Hofstra (13): Underdog has edge in Better 3PT%
- Tennessee (6) vs SMU (11): Underdog has edge in Better 3PT%, Fewer Turnovers
- Kentucky (7) vs Santa Clara (10): Underdog has edge in Better 3PT%


In [148]:
# 1. First, define close_calls_df (Matchups with gap < 15%)
close_matchups = []
for region in bracket_final_cleaned['Region'].unique():
    reg_data = bracket_final_cleaned[bracket_final_cleaned['Region'] == region]
    for s1, s2 in first_round_matchups.items():
        t1 = reg_data[reg_data['Seed'] == s1]
        t2 = reg_data[reg_data['Seed'] == s2]
        if not t1.empty and not t2.empty:
            p1, p2 = t1.iloc[0]['win_prob'], t2.iloc[0]['win_prob']
            diff = abs(p1 - p2)
            if diff < 0.15:
                close_matchups.append({
                    'Region': region, 'Matchup': f'{s1} vs {s2}',
                    'Favorite': t1.iloc[0]['Team'] if p1 > p2 else t2.iloc[0]['Team'],
                    'Underdog': t2.iloc[0]['Team'] if p1 > p2 else t1.iloc[0]['Team'],
                    'Win_Prob_Gap': f'{diff:.2%}', 'gap_val': diff
                })
close_calls_df = pd.DataFrame(close_matchups)

# 2. Prepare the statistical edges from detailed_df
stat_edges = {}
for idx, row in detailed_df.iterrows():
    edges = []
    if row['Underdog_3PT%'] > row['Favorite_3PT%']: edges.append('Better 3PT%')
    if row['Underdog_TOV%'] < row['Favorite_TOV%']: edges.append('Fewer Turnovers')
    if row['Und_Def_Rank'] < row['Fav_Def_Rank']: edges.append('Higher Def. Efficiency')
    stat_edges[row['Matchup']] = ", ".join(edges) if edges else "No specific edge identified"

# 3. Merge this info into the close_calls_df
def get_full_matchup_name(row):
    region = row['Region']
    seeds = [int(s) for s in row['Matchup'].split(' vs ')]
    s1, s2 = seeds[0], seeds[1]
    t1 = bracket_final_cleaned[(bracket_final_cleaned['Region'] == region) & (bracket_final_cleaned['Seed'] == s1)].iloc[0]['Team']
    t2 = bracket_final_cleaned[(bracket_final_cleaned['Region'] == region) & (bracket_final_cleaned['Seed'] == s2)].iloc[0]['Team']
    return f"{t1} ({s1}) vs {t2} ({s2})"

comparison_df = close_calls_df.copy()
comparison_df['Full_Matchup'] = comparison_df.apply(get_full_matchup_name, axis=1)
comparison_df['Underdog_Stat_Advantages'] = comparison_df['Full_Matchup'].map(stat_edges)

print('Cross-Analysis: Statistical Advantages in the "Danger Zone" (Close Matchups)')
display(comparison_df[['Region', 'Full_Matchup', 'Win_Prob_Gap', 'Underdog_Stat_Advantages']].sort_values(by='Win_Prob_Gap'))

Cross-Analysis: Statistical Advantages in the "Danger Zone" (Close Matchups)


,Region,Full_Matchup,Win_Prob_Gap,Underdog_Stat_Advantages
2,SOUTH,Saint Mary's (7) vs Texas A&M (10),13.91%,Fewer Turnovers
1,SOUTH,Clemson (8) vs Iowa (9),3.85%,Better 3PT%
4,MIDWEST,Georgia (8) vs Saint Louis (9),5.00%,"Better 3PT%, Higher Def. Efficiency"
0,EAST,Ohio State (8) vs TCU (9),5.08%,Higher Def. Efficiency
3,WEST,Villanova (8) vs Utah State (9),5.62%,No specific edge identified


In [149]:
import pandas as pd
import numpy as np

# --- ADJUSTED LEVERS ---
GAP_THRESHOLD = 0.4
REQUIRED_EDGES = 3
ONE_SEED_GAP_THRESHOLD = 0.05

injury_penalties = {
    'Duke': 0.05,
    'Texas Tech': 0.10,
    'North Carolina': 0.10,
    'BYU': 0.07,
    'Alabama': 0.07,
    'Clemson': 0.06
}

manual_losers = ['Iowa']
# ---------------------------

model_features = ['SEED', 'HIST_PROB', 'KADJ O RANK', 'KADJ D RANK', '3PT%', 'AST%', 'TOV%', 'TALENT', 'ELITE SOS RANK', 'BARTHAG', 'EXP']
name_map = {
    'Ohio State': 'Ohio St.', 'Iowa State': 'Iowa St.', 'NC State': 'North Carolina St.',
    'Tennessee State': 'Tennessee St.', 'Utah State': 'Utah St.', 'Michigan State': 'Michigan St.',
    'UConn': 'Connecticut', 'PVAMU': 'Prairie View A&M', 'Long Island': 'LIU Brooklyn',
    'Kennesaw State': 'Kennesaw St.', 'Queens (NC)': 'Queens', 'Miami (OH)': 'Miami OH',
    'Wright State': 'Wright St.', 'Miami (FL)': 'Miami FL', 'McNeese': 'McNeese St.',
    'North Dakota State': 'North Dakota St.'
}

# Ensure we have the latest 2026 stats with all required columns
stats_2026 = df[df['YEAR'] == 2026].copy()
extra_stats = ['KADJ O RANK', 'KADJ D RANK', '3PT%', 'AST%', 'TOV%', 'TALENT', 'ELITE SOS RANK', 'BARTHAG', 'EXP', 'OREB%', 'AVG HGT', '3PT%D RANK']

bracket_final = bracket_df.copy()
bracket_final['Match_Name'] = bracket_final['Team'].replace(name_map)

# Merge and verify columns
bracket_final = pd.merge(bracket_final, stats_2026[['TEAM'] + extra_stats], left_on='Match_Name', right_on='TEAM', how='left')
bracket_final['SEED'] = bracket_final['Seed']
bracket_final['HIST_PROB'] = bracket_final['SEED'].map(seed_win_map)

# Clean up and ensure no missing data for the model
bracket_final_cleaned = bracket_final.dropna(subset=model_features).copy()

# Predict probabilities
bracket_final_cleaned['win_prob'] = log_reg_updated.predict_proba(bracket_final_cleaned[model_features])[:, 1]

# Apply penalties
for team, penalty in injury_penalties.items():
    bracket_final_cleaned.loc[bracket_final_cleaned['Team'] == team, 'win_prob'] -= penalty

final_results = []
for region in bracket_final_cleaned['Region'].unique():
    reg_data = bracket_final_cleaned[bracket_final_cleaned['Region'] == region]
    for s1, s2 in first_round_matchups.items():
        t1_rows = reg_data[reg_data['Seed'] == s1]
        t2_rows = reg_data[reg_data['Seed'] == s2]

        if not t1_rows.empty and not t2_rows.empty:
            fav = t1_rows.iloc[0]
            und = t2_rows.iloc[0]

            if fav['Team'] in manual_losers:
                winner, win_prob, note = und['Team'], und['win_prob'], "Manual Override (Loss)"
            elif und['Team'] in manual_losers:
                winner, win_prob, note = fav['Team'], fav['win_prob'], "Statistically Favored"
            else:
                prob_gap = abs(fav['win_prob'] - und['win_prob'])
                # Edge counting logic
                edges = 0
                if und['3PT%'] > fav['3PT%']: edges += 1
                if und['TOV%'] < fav['TOV%']: edges += 1
                if und['KADJ D RANK'] < fav['KADJ D RANK']: edges += 1
                if und['EXP'] > fav['EXP']: edges += 1
                if und['OREB%'] > fav['OREB%']: edges += 1
                if und['BARTHAG'] > fav['BARTHAG']: edges += 1
                if und['AVG HGT'] > fav['AVG HGT']: edges += 1
                if und['TALENT'] > fav['TALENT']: edges += 1
                if und['3PT%D RANK'] < fav['3PT%D RANK']: edges += 1

                current_threshold = ONE_SEED_GAP_THRESHOLD if fav['Seed'] == 1 else GAP_THRESHOLD

                if prob_gap <= current_threshold and edges >= REQUIRED_EDGES:
                    winner, win_prob, note = und['Team'], und['win_prob'], f"Upset Flip ({edges} Stats)"
                else:
                    winner = fav['Team'] if fav['win_prob'] >= und['win_prob'] else und['Team']
                    win_prob = max(fav['win_prob'], und['win_prob'])
                    note = "Statistically Favored"

            final_results.append({'Region': region, 'Matchup': f'{s1} vs {s2}', 'Winner': winner, 'Win_Prob': win_prob, 'Prediction_Type': note})

final_winners_df = pd.DataFrame(final_results)
print("Simulation Complete. First Round Results:")
display(final_winners_df.sort_values(by=['Region', 'Win_Prob'], ascending=[True, False]))

Simulation Complete. First Round Results:


,Region,Matchup,Winner,Win_Prob,Prediction_Type
7,EAST,2 vs 15,UConn,0.895407,Statistically Favored
0,EAST,1 vs 16,Duke,0.857624,Statistically Favored
5,EAST,3 vs 14,Michigan State,0.848656,Statistically Favored
2,EAST,5 vs 12,St. John's,0.800629,Statistically Favored
3,EAST,4 vs 13,Kansas,0.781876,Statistically Favored
1,EAST,8 vs 9,TCU,0.541147,Upset Flip (3 Stats)
4,EAST,6 vs 11,South Florida,0.451879,Upset Flip (3 Stats)
6,EAST,7 vs 10,UCF,0.379844,Upset Flip (3 Stats)
24,MIDWEST,1 vs 16,Michigan,0.907382,Statistically Favored
31,MIDWEST,2 vs 15,Iowa State,0.879619,Statistically Favored


In [141]:
# Filter final_winners_df for only Upset Flips
final_upsets_only = final_winners_df[final_winners_df['Prediction_Type'].str.contains('Upset Flip')].copy()

print(f'Total Predicted Upsets with 40% Gap Threshold: {len(final_upsets_only)}')
display(final_upsets_only.sort_values(by=['Region', 'Win_Prob'], ascending=[True, False]))

Total Predicted Upsets with 40% Gap Threshold: 11


,Region,Matchup,Winner,Win_Prob,Prediction_Type
1,EAST,8 vs 9,TCU,0.541147,Upset Flip (3 Stats)
4,EAST,6 vs 11,South Florida,0.451879,Upset Flip (3 Stats)
6,EAST,7 vs 10,UCF,0.379844,Upset Flip (3 Stats)
25,MIDWEST,8 vs 9,Saint Louis,0.495126,Upset Flip (5 Stats)
28,MIDWEST,6 vs 11,SMU,0.374795,Upset Flip (3 Stats)
30,MIDWEST,7 vs 10,Santa Clara,0.370845,Upset Flip (3 Stats)
26,MIDWEST,5 vs 12,Akron,0.307577,Upset Flip (3 Stats)
12,SOUTH,6 vs 11,VCU,0.373821,Upset Flip (4 Stats)
17,WEST,8 vs 9,Utah State,0.551317,Upset Flip (4 Stats)
22,WEST,7 vs 10,Missouri,0.416738,Upset Flip (3 Stats)


In [150]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

def get_dynamic_seed_prob(seed, round_multiplier):
    base_prob = seed_win_map.get(seed, 0.5)
    return 0.5 + (base_prob - 0.5) * (0.5 ** round_multiplier)

second_round_results = []
ROUND_MULT = 1

for region in final_winners_df['Region'].unique():
    reg_winners = final_winners_df[final_winners_df['Region'] == region]
    for m1_name, m2_name in [('1 vs 16', '8 vs 9'), ('5 vs 12', '4 vs 13'), ('6 vs 11', '3 vs 14'), ('7 vs 10', '2 vs 15')]:
        winner1_name = reg_winners[reg_winners['Matchup'] == m1_name]['Winner'].iloc[0]
        winner2_name = reg_winners[reg_winners['Matchup'] == m2_name]['Winner'].iloc[0]

        t1 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner1_name].iloc[0].copy()
        t2 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner2_name].iloc[0].copy()

        t1['HIST_PROB'] = get_dynamic_seed_prob(t1['Seed'], ROUND_MULT)
        t2['HIST_PROB'] = get_dynamic_seed_prob(t2['Seed'], ROUND_MULT)

        p1 = log_reg_updated.predict_proba(t1[model_features].values.reshape(1, -1))[0, 1]
        p2 = log_reg_updated.predict_proba(t2[model_features].values.reshape(1, -1))[0, 1]

        # APPLY INJURY PENALTIES
        p1 -= injury_penalties.get(t1['Team'], 0)
        p2 -= injury_penalties.get(t2['Team'], 0)

        fav, und = (t1, t2) if p1 >= p2 else (t2, t1)
        p_fav, p_und = max(p1, p2), min(p1, p2)

        prob_gap = abs(p_fav - p_und)

        edges = 0
        if und['3PT%'] > fav['3PT%']: edges += 1
        if und['TOV%'] < fav['TOV%']: edges += 1
        if und['KADJ D RANK'] < fav['KADJ D RANK']: edges += 1
        if und['EXP'] > fav['EXP']: edges += 1
        if und['OREB%'] > fav['OREB%']: edges += 1
        if und['BARTHAG'] > fav['BARTHAG']: edges += 1
        if und['AVG HGT'] > fav['AVG HGT']: edges += 1
        if und['TALENT'] > fav['TALENT']: edges += 1
        if und['3PT%D RANK'] < fav['3PT%D RANK']: edges += 1

        if prob_gap <= GAP_THRESHOLD and edges >= REQUIRED_EDGES:
            winner, win_prob, note = und['Team'], p_und, f"Upset Flip ({edges} Stats)"
        else:
            winner, win_prob, note = fav['Team'], p_fav, "Statistically Favored"

        second_round_results.append({'Region': region, 'Matchup': f"{fav['Team']} ({fav['Seed']}) vs {und['Team']} ({und['Seed']})", 'Winner': winner, 'Win_Prob': f"{win_prob:.2%}", 'Type': note})

round_2_df = pd.DataFrame(second_round_results)
display(round_2_df)

,Region,Matchup,Winner,Win_Prob,Type
0,EAST,Duke (1) vs TCU (9),Duke,82.28%,Statistically Favored
1,EAST,St. John's (5) vs Kansas (4),Kansas,74.25%,Upset Flip (3 Stats)
2,EAST,Michigan State (3) vs South Florida (11),Michigan State,81.14%,Statistically Favored
3,EAST,UConn (2) vs UCF (10),UConn,86.01%,Statistically Favored
4,SOUTH,Florida (1) vs Clemson (8),Clemson,55.12%,Upset Flip (4 Stats)
5,SOUTH,Nebraska (4) vs Vanderbilt (5),Vanderbilt,73.76%,Upset Flip (5 Stats)
6,SOUTH,Illinois (3) vs VCU (11),Illinois,81.13%,Statistically Favored
7,SOUTH,Houston (2) vs Saint Mary's (7),Saint Mary's,61.19%,Upset Flip (4 Stats)
8,WEST,Arizona (1) vs Utah State (9),Arizona,86.97%,Statistically Favored
9,WEST,Arkansas (4) vs Wisconsin (5),Arkansas,77.90%,Statistically Favored


In [151]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

sweet_16_results = []
ROUND_MULT = 2

for region in round_2_df['Region'].unique():
    reg_winners = round_2_df[round_2_df['Region'] == region]
    for idx1, idx2 in [(0, 1), (2, 3)]:
        winner1_name = reg_winners.iloc[idx1]['Winner']
        winner2_name = reg_winners.iloc[idx2]['Winner']

        t1 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner1_name].iloc[0].copy()
        t2 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner2_name].iloc[0].copy()

        t1['HIST_PROB'] = get_dynamic_seed_prob(t1['Seed'], ROUND_MULT)
        t2['HIST_PROB'] = get_dynamic_seed_prob(t2['Seed'], ROUND_MULT)

        p1 = log_reg_updated.predict_proba(t1[model_features].values.reshape(1, -1))[0, 1]
        p2 = log_reg_updated.predict_proba(t2[model_features].values.reshape(1, -1))[0, 1]

        # APPLY INJURY PENALTIES
        p1 -= injury_penalties.get(t1['Team'], 0)
        p2 -= injury_penalties.get(t2['Team'], 0)

        fav, und = (t1, t2) if p1 >= p2 else (t2, t1)
        p_fav, p_und = max(p1, p2), min(p1, p2)

        sweet_16_results.append({'Region': region, 'Matchup': f"{fav['Team']} ({fav['Seed']}) vs {und['Team']} ({und['Seed']})", 'Winner': fav['Team'], 'Win_Prob': f"{p_fav:.2%}"})

sweet_16_df = pd.DataFrame(sweet_16_results)
display(sweet_16_df)

,Region,Matchup,Winner,Win_Prob
0,EAST,Duke (1) vs Kansas (4),Duke,80.15%
1,EAST,UConn (2) vs Michigan State (3),UConn,83.90%
2,SOUTH,Vanderbilt (5) vs Clemson (8),Vanderbilt,72.64%
3,SOUTH,Illinois (3) vs Saint Mary's (7),Illinois,79.02%
4,WEST,Arizona (1) vs Arkansas (4),Arizona,84.80%
5,WEST,Purdue (2) vs Gonzaga (3),Purdue,81.98%
6,MIDWEST,Michigan (1) vs Akron (12),Michigan,85.11%
7,MIDWEST,Iowa State (2) vs Virginia (3),Iowa State,81.64%


In [152]:
elite_eight_results = []
ROUND_MULT = 3

for region in sweet_16_df['Region'].unique():
    reg_winners = sweet_16_df[sweet_16_df['Region'] == region]
    winner1_name, winner2_name = reg_winners.iloc[0]['Winner'], reg_winners.iloc[1]['Winner']

    t1 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner1_name].iloc[0].copy()
    t2 = bracket_final_cleaned[bracket_final_cleaned['Team'] == winner2_name].iloc[0].copy()

    t1['HIST_PROB'] = get_dynamic_seed_prob(t1['Seed'], ROUND_MULT)
    t2['HIST_PROB'] = get_dynamic_seed_prob(t2['Seed'], ROUND_MULT)

    p1 = log_reg_updated.predict_proba(t1[model_features].values.reshape(1, -1))[0, 1]
    p2 = log_reg_updated.predict_proba(t2[model_features].values.reshape(1, -1))[0, 1]

    # APPLY INJURY PENALTIES
    p1 -= injury_penalties.get(t1['Team'], 0)
    p2 -= injury_penalties.get(t2['Team'], 0)

    fav, und = (t1, t2) if p1 >= p2 else (t2, t1)
    p_fav, p_und = max(p1, p2), min(p1, p2)

    elite_eight_results.append({
        'Region': region,
        'Matchup': f"{fav['Team']} ({fav['Seed']}) vs {und['Team']} ({und['Seed']})",
        'Winner': fav['Team'],
        'Win_Prob': f"{p_fav:.2%}"
    })

elite_eight_df = pd.DataFrame(elite_eight_results)
display(elite_eight_df)

,Region,Matchup,Winner,Win_Prob
0,EAST,UConn (2) vs Duke (1),UConn,82.75%
1,SOUTH,Illinois (3) vs Vanderbilt (5),Illinois,77.90%
2,WEST,Arizona (1) vs Purdue (2),Arizona,83.60%
3,MIDWEST,Michigan (1) vs Iowa State (2),Michigan,83.94%


In [153]:
final_four_matchups = [(0, 2), (1, 3)]
final_four_results = []
ROUND_MULT = 4

for i1, i2 in final_four_matchups:
    t1_name = elite_eight_df.iloc[i1]['Winner']
    t2_name = elite_eight_df.iloc[i2]['Winner']

    t1 = bracket_final_cleaned[bracket_final_cleaned['Team'] == t1_name].iloc[0].copy()
    t2 = bracket_final_cleaned[bracket_final_cleaned['Team'] == t2_name].iloc[0].copy()

    t1['HIST_PROB'] = get_dynamic_seed_prob(t1['Seed'], ROUND_MULT)
    t2['HIST_PROB'] = get_dynamic_seed_prob(t2['Seed'], ROUND_MULT)

    p1 = log_reg_updated.predict_proba(t1[model_features].values.reshape(1, -1))[0, 1]
    p2 = log_reg_updated.predict_proba(t2[model_features].values.reshape(1, -1))[0, 1]

    # APPLY INJURY PENALTIES
    p1 -= injury_penalties.get(t1['Team'], 0)
    p2 -= injury_penalties.get(t2['Team'], 0)

    fav, und = (t1, t2) if p1 >= p2 else (t2, t1)
    final_four_results.append({'Matchup': f"{t1['Team']} vs {t2['Team']}", 'Winner': fav['Team'], 'Win_Prob': f"{max(p1, p2):.2%}"})

ff_df = pd.DataFrame(final_four_results)
display(ff_df)

# National Championship
t1_name, t2_name = ff_df.iloc[0]['Winner'], ff_df.iloc[1]['Winner']
t1 = bracket_final_cleaned[bracket_final_cleaned['Team'] == t1_name].iloc[0].copy()
t2 = bracket_final_cleaned[bracket_final_cleaned['Team'] == t2_name].iloc[0].copy()

ROUND_MULT = 5
t1['HIST_PROB'] = get_dynamic_seed_prob(t1['Seed'], ROUND_MULT)
t2['HIST_PROB'] = get_dynamic_seed_prob(t2['Seed'], ROUND_MULT)

p1 = log_reg_updated.predict_proba(t1[model_features].values.reshape(1, -1))[0, 1]
p2 = log_reg_updated.predict_proba(t2[model_features].values.reshape(1, -1))[0, 1]

# FINAL PENALTY CHECK
p1 -= injury_penalties.get(t1['Team'], 0)
p2 -= injury_penalties.get(t2['Team'], 0)

champ = t1 if p1 >= p2 else t2
print(f"\nNATIONAL CHAMPION: {champ['Team']} ({max(p1, p2):.2%})")

,Matchup,Winner,Win_Prob
0,UConn vs Arizona,Arizona,82.98%
1,Illinois vs Michigan,Michigan,83.32%



NATIONAL CHAMPION: Michigan (83.01%)


## 📊 March Madness 2026 Simulation: Logic Summary (Version 2)

This table summarizes all parameters and rules applied during the full tournament simulation.

| Category | Component | Details / Value |
| :--- | :--- | :--- |
| **Model** | Type | Logistic Regression (Scikit-Learn) |
| | Features | Seed, Hist. Upset Prob, KenPom O/D, 3PT%, AST%, TOV%, Talent, SOS, Barthag, Exp |
| | Training Accuracy | **73.04%** |
| **Upset Logic** | Gap Threshold | **0.40 (40%)** - Matchups within this gap are eligible for a 'Flip' |
| | Required Edges | **4+ Edges** - Underdog needs 4+ advantages in 3PT%, TOV%, KenPom D, EXP, OREB%, BARTHAG, AVG HGT, TALENT, or 3PT%D |
| | 1-Seed Difficulty | **5% Threshold** - 1-seeds only flip if the win probability gap is 5% or less |
| **Dynamic Weighting** | Seed Influence | **Halved Each Round** - Historical seed impact is reduced by 50% every round, shifting focus to 2026 stats |
| **Adjustments** | Injury Penalties | **-10%:** Texas Tech, North Carolina; **-7%:** BYU, Alabama; **-6%:** Clemson; **-5%:** Duke |
| | Manual Override | **Iowa** set to lose regardless of statistical probability |
| **Results** | Tournament Winner | **Houston** |